In [2]:
from dotenv import load_dotenv
load_dotenv()


## import required Liberaries
from pydantic import BaseModel, Field
from typing import Literal
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain.tools import tool
from langchain_community.utilities import GoogleSerperAPIWrapper

## define the class
class FlowState(BaseModel):
    question : str = Field(description="User asked Question")
    category : Literal["coding", "google search", "weather"] = Field(default="google search")
    ans : str = Field(default="")

class QuestionCategory(BaseModel):
    category : Literal["coding", "google search", "weather"] = Field(default="google search", description="Question category")

llm = ChatGroq(model="llama-3.3-70b-versatile")

## define functions / nodes
def check_question_category(state:FlowState) -> FlowState:
    st_llm = llm.with_structured_output(QuestionCategory)
    res = st_llm.invoke(f"Categorize my question {state.question} if not sure, just say google search")
    state.category = res.category
    return state

def route(state: FlowState) -> Literal["coding", "google search", "weather"]:
    return state.category

def coding_node(state:FlowState) -> FlowState:
    res = llm.invoke(f"You are a coding expert. {state.question}")
    state.ans = res.content
    return state

def Google_search_node(state:FlowState) -> FlowState:
    state.ans = "This is a google search node"
    return state

def weather_node(state:FlowState) -> FlowState:
    state.ans = "this is weather node"
    return state

graph = StateGraph(FlowState)

graph.add_node("check question category", check_question_category)
graph.add_node("coding", coding_node)
graph.add_node("google search", Google_search_node)
graph.add_node("weather", weather_node)

graph.add_edge(START, "check question category")
graph.add_conditional_edges("check question category", route)
graph.add_edge("coding", END)
graph.add_edge("google search", END)
graph.add_edge("weather", END)

graph.compile

from IPython.display import Image
Image(graph.get_graph().draw_mermaid_png())

AttributeError: 'StateGraph' object has no attribute 'get_graph'